# 02 — Damage Detection

This notebook replicates the **damage detection** ML module from the `building_analyzer` repository.

It covers:
1. Installing dependencies and cloning the repository
2. Understanding the dataset interface (COCO-format annotations)
3. Instantiating anchor-free and two-stage detectors
4. Running a short training loop on a synthetic dataset
5. Running inference with NMS post-processing
6. Visualising detection results
7. Computing mAP evaluation metrics


In [ ]:
!pip install torch torchvision albumentations Pillow numpy matplotlib --quiet

In [ ]:
import subprocess, sys, os
if not os.path.exists('building_analyzer'):
    subprocess.run(['git', 'clone', 'https://github.com/Tripoid/building_analyzer.git'], check=True)
REPO_ROOT = os.path.abspath('building_analyzer')
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print('Ready. REPO_ROOT =', REPO_ROOT)

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from torch.utils.data import Dataset, DataLoader

from ml.common.registry import ModelRegistry
from ml.common.metrics import compute_map
from ml.damage_detection.dataset import DAMAGE_CLASS_NAMES
from ml.damage_detection.model import AnchorFreeDamageDetector, TwoStageDetector
from ml.damage_detection.inference import DamageDetectionInferencer, DamageDetectionInferencerConfig
from ml.damage_detection.train import DamageDetectionTrainer, DamageDetectionTrainerConfig
from ml.damage_detection.utils import apply_nms, draw_detections

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
NUM_CLASSES = len(DAMAGE_CLASS_NAMES)
print(f'Device: {DEVICE}  |  Damage classes: {DAMAGE_CLASS_NAMES}')

## Model architectures

In [ ]:
print('Registered detection models:', ModelRegistry.list_models(namespace='detection'))

anchor_free = AnchorFreeDamageDetector(
    num_classes=NUM_CLASSES, class_names=DAMAGE_CLASS_NAMES, fpn_channels=128
)
two_stage = TwoStageDetector(
    num_classes=NUM_CLASSES, class_names=DAMAGE_CLASS_NAMES
)

print(f'AnchorFree params: {sum(p.numel() for p in anchor_free.parameters()):,}')
print(f'TwoStage params:   {sum(p.numel() for p in two_stage.parameters()):,}')

## Synthetic detection dataset

In [ ]:
from ml.common.transforms import get_detection_transforms

class SyntheticDetDataset(Dataset):
    """Random images with random bounding boxes for quick experimentation."""
    def __init__(self, n=64, image_size=(128, 128), num_classes=5, max_boxes=3):
        self.n = n
        self.h, self.w = image_size
        self.num_classes = num_classes
        self.max_boxes = max_boxes

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        image_t = torch.rand(3, self.h, self.w)
        nb = np.random.randint(1, self.max_boxes + 1)
        boxes, labels = [], []
        for _ in range(nb):
            x1 = np.random.randint(0, self.w - 20)
            y1 = np.random.randint(0, self.h - 20)
            x2 = np.random.randint(x1 + 10, min(x1 + 60, self.w))
            y2 = np.random.randint(y1 + 10, min(y1 + 60, self.h))
            boxes.append([x1, y1, x2, y2])
            labels.append(np.random.randint(0, self.num_classes))
        target = {
            'boxes': torch.tensor(boxes, dtype=torch.float32),
            'labels': torch.tensor(labels, dtype=torch.int64),
        }
        return image_t, target

def collate_fn(batch):
    images, targets = zip(*batch)
    return torch.stack(images), list(targets)

train_ds = SyntheticDetDataset(n=128)
val_ds   = SyntheticDetDataset(n=32)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, collate_fn=collate_fn)
print('Dataset ready.')

## Training

In [ ]:
model = AnchorFreeDamageDetector(
    num_classes=NUM_CLASSES, class_names=DAMAGE_CLASS_NAMES, fpn_channels=64
)
config = DamageDetectionTrainerConfig(
    output_dir='/tmp/det_checkpoints',
    num_epochs=3,
    learning_rate=1e-3,
    device=DEVICE,
    mixed_precision=(DEVICE == 'cuda'),
    log_every_n_steps=5,
    score_threshold=0.3,
)
trainer = DamageDetectionTrainer(model, config)
history = trainer.train(train_loader, val_loader)
print('Training complete! Loss history:', [f"{v:.4f}" for v in history['train_loss']])

## Inference and visualisation

In [ ]:
cfg = DamageDetectionInferencerConfig(
    device=DEVICE, image_size=(128, 128), score_threshold=0.2, max_detections=20
)
inferencer = DamageDetectionInferencer(model=model, config=cfg)

# Generate a random test image
test_img_np = np.random.randint(0, 255, (256, 256, 3), dtype=np.uint8)
prediction = inferencer.predict_from_array(test_img_np)

print(f'Detected {prediction.num_detections} damage instances')
for inst in prediction.result.instances[:5]:
    cls_name = DAMAGE_CLASS_NAMES[inst.label] if inst.label < NUM_CLASSES else 'unknown'
    print(f'  {cls_name}: score={inst.score:.3f}  box={[round(c,1) for c in inst.box]}')

# Visualise
vis = prediction.visualize(test_img_np)
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.imshow(test_img_np); plt.title('Input'); plt.axis('off')
plt.subplot(1, 2, 2); plt.imshow(vis); plt.title(f'Detections ({prediction.num_detections})'); plt.axis('off')
plt.tight_layout(); plt.show()

## NMS post-processing

In [ ]:
# Demonstrate NMS directly
boxes_demo  = torch.tensor([[0, 0, 50, 50], [5, 5, 45, 45], [200, 200, 250, 250]], dtype=torch.float32)
labels_demo = torch.tensor([0, 0, 1])
scores_demo = torch.tensor([0.9, 0.8, 0.7])

kept_boxes, kept_labels, kept_scores = apply_nms(
    boxes_demo, labels_demo, scores_demo,
    score_threshold=0.5, iou_threshold=0.3
)
print(f'Before NMS: {boxes_demo.shape[0]} boxes')
print(f'After NMS:  {kept_boxes.shape[0]} boxes')
print('Kept boxes:', kept_boxes.tolist())

## mAP evaluation

In [ ]:
model.eval()
all_preds, all_gts = [], []
with torch.no_grad():
    for images, targets in val_loader:
        images = images.to(DEVICE)
        results = model.predict(images, score_threshold=0.0)
        for r, t in zip(results, targets):
            all_preds.append({
                'boxes':  [inst.box for inst in r.instances],
                'labels': [inst.label for inst in r.instances],
                'scores': [inst.score for inst in r.instances],
            })
            all_gts.append({'boxes': t['boxes'].tolist(), 'labels': t['labels'].tolist()})

metrics = compute_map(all_preds, all_gts, iou_threshold=0.5, num_classes=NUM_CLASSES)
print(f'mAP@0.5 = {metrics["map"]:.4f}')
print('Per-class AP:')
for cls_id, ap in metrics['per_class_ap'].items():
    print(f'  {DAMAGE_CLASS_NAMES[cls_id] if cls_id < NUM_CLASSES else cls_id}: {ap:.4f}')